# Iceberg Classification — Integrated Training Pipeline

In [1]:
from pathlib import Path
import subprocess
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models

from sklearn.model_selection import StratifiedKFold

from torch.utils.data import Dataset, DataLoader

from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomResizedCrop,
    InterpolationMode,
)


SEED = 42
BATCH_SIZE = 32
NUM_FOLDS = 5

MAX_EPOCHS = 30
INITIAL_LR = 0.001
MIN_LR = 0.00001
PATIENCE = 8

NUM_WORKERS = 2

# Set False if pretrained EfficientNet weights are unavailable.
USE_PRETRAINED = True

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: Tesla T4


In [2]:
INPUT_DIR = Path(
    "/kaggle/input/competitions/statoil-iceberg-classifier-challenge"
)

WORK_DIR = Path(
    "/kaggle/working/iceberg-data"
)

TRAIN_PATH = (
    WORK_DIR / "data/processed/train.json"
)

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Extract the training data only if it is not already available.
if not TRAIN_PATH.exists():

    subprocess.run(
        [
            "7z",
            "x",
            str(INPUT_DIR / "train.json.7z"),
            f"-o{WORK_DIR}",
            "-y",
        ],
        check=True,
    )

train_df = pd.read_json(
    TRAIN_PATH
)

print("Training samples:", len(train_df))

Training samples: 1604


In [3]:
def process_train_dataframe(df):
    """Convert the raw training DataFrame into NumPy arrays."""

    df = df.copy()

    # Convert incidence angles to numeric values and keep missing entries as NaN.
    angles = pd.to_numeric(
        df["inc_angle"],
        errors="coerce",
    ).to_numpy(
        dtype=np.float32
    ).reshape(-1, 1)

    # Record whether each incidence angle was originally missing.
    angle_missing = np.isnan(
        angles
    ).astype(np.float32)

    # Reshape both SAR bands from 5625 values to 75 x 75 images.
    band_1 = np.stack([
        np.asarray(
            band,
            dtype=np.float32
        ).reshape(75, 75)

        for band in df["band_1"]
    ])

    band_2 = np.stack([
        np.asarray(
            band,
            dtype=np.float32
        ).reshape(75, 75)

        for band in df["band_2"]
    ])

    # Use the average of the two SAR bands as the third channel.
    band_3 = (
        band_1 + band_2
    ) / 2.0

    # PyTorch uses channel-first format: [N, C, H, W].
    images = np.stack(
        [
            band_1,
            band_2,
            band_3
        ],
        axis=1,
    ).astype(np.float32)

    ids = df[
        "id"
    ].to_numpy()

    labels = df[
        "is_iceberg"
    ].to_numpy(
        dtype=np.float32
    )

    return (
        ids,
        images,
        angles,
        angle_missing,
        labels
    )


(
    train_ids,
    X_train_img,
    X_train_ang,
    X_train_ang_missing,
    Y_train,
) = process_train_dataframe(
    train_df
)

print("Image shape:", X_train_img.shape)
print("Angle shape:", X_train_ang.shape)
print("Label shape:", Y_train.shape)

Image shape: (1604, 3, 75, 75)
Angle shape: (1604, 1)
Label shape: (1604,)


In [4]:
splitter = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=SEED,
)

fold_numbers = np.full(
    len(train_df),
    fill_value=-1,
    dtype=np.int64,
)

for fold, (_, val_indices) in enumerate(
    splitter.split(
        train_df,
        train_df["is_iceberg"]
    )
):

    fold_numbers[
        val_indices
    ] = fold

train_df[
    "fold"
] = fold_numbers


print("Samples in each fold:")

print(
    train_df[
        "fold"
    ].value_counts().sort_index()
)

print(
    "\nLabel distribution:"
)

print(
    pd.crosstab(
        train_df["fold"],
        train_df["is_iceberg"],
    )
)

Samples in each fold:
fold
0    321
1    321
2    321
3    321
4    320
Name: count, dtype: int64

Label distribution:
is_iceberg    0    1
fold                
0           171  150
1           170  151
2           170  151
3           170  151
4           170  150


In [5]:
class IcebergDataset(Dataset):

    """Dataset for SAR images, incidence angles, labels, and IDs."""

    def __init__(
        self,
        images,
        angles,
        angle_missing,
        ids,
        indices,
        angle_mean,
        angle_std,
        labels=None,
        transform=None,
    ):

        self.images = images
        self.angles = angles
        self.angle_missing = angle_missing
        self.ids = ids

        self.indices = np.asarray(
            indices
        )

        self.angle_mean = float(
            angle_mean
        )

        self.angle_std = float(
            angle_std
        )

        if self.angle_std <= 0:

            raise ValueError(
                "angle_std must be greater than 0"
            )

        self.labels = labels
        self.transform = transform


    def __len__(self):

        return len(
            self.indices
        )


    def __getitem__(self, index):

        original_index = (
            self.indices[index]
        )

        image = torch.from_numpy(
            self.images[
                original_index
            ].copy()
        ).float()

        if self.transform is not None:

            image = self.transform(
                image
            )

        angle = float(
            self.angles[
                original_index,
                0
            ]
        )

        missing = float(
            self.angle_missing[
                original_index,
                0
            ]
        )

        # Fill missing angles with the training-fold mean.
        if np.isnan(angle):

            angle = self.angle_mean

        # Standardize using training-fold statistics.
        angle = (
            angle - self.angle_mean
        ) / self.angle_std

        sample = {
            "image": image,

            "angle": torch.tensor(
                [angle],
                dtype=torch.float32,
            ),

            "angle_missing": torch.tensor(
                [missing],
                dtype=torch.float32,
            ),

            "id": str(
                self.ids[
                    original_index
                ]
            ),
        }

        if self.labels is not None:

            sample[
                "target"
            ] = torch.tensor(
                self.labels[
                    original_index
                ],
                dtype=torch.float32,
            )

        return sample

In [6]:
class RandomRotate90:

    def __call__(self, image):

        k = int(
            torch.randint(
                0,
                4,
                (1,)
            ).item()
        )

        return torch.rot90(
            image,
            k=k,
            dims=(-2, -1)
        )


def build_train_transform(
    image_mean,
    image_std,
    augmentation
):

    normalize = Normalize(
        mean=image_mean.tolist(),
        std=image_std.tolist(),
    )

    if augmentation == "baseline":

        return Compose([
            normalize
        ])

    if augmentation == "hflip":

        return Compose([
            RandomHorizontalFlip(
                p=0.5
            ),
            normalize,
        ])

    if augmentation == "vflip":

        return Compose([
            RandomVerticalFlip(
                p=0.5
            ),
            normalize,
        ])

    if augmentation == "hvflip":

        return Compose([
            RandomHorizontalFlip(
                p=0.5
            ),
            RandomVerticalFlip(
                p=0.5
            ),
            normalize,
        ])

    if augmentation == "rotation":

        return Compose([
            RandomRotate90(),
            normalize,
        ])

    crop_scales = {
        "crop_light": (0.9, 1.0),
        "crop_medium": (0.8, 1.0),
    }

    if augmentation in crop_scales:

        return Compose([
            RandomResizedCrop(
                size=(75, 75),
                scale=crop_scales[augmentation],
                ratio=(1.0, 1.0),
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            ),
            normalize,
        ])

    raise ValueError(
        f"Unknown augmentation: {augmentation}"
    )


def build_val_transform(
    image_mean,
    image_std
):

    return Normalize(
        mean=image_mean.tolist(),
        std=image_std.tolist(),
    )

In [7]:
def create_fold_loaders(
    fold,
    augmentation="baseline",
    batch_size=BATCH_SIZE,
    seed=SEED,
):

    fold_array = train_df[
        "fold"
    ].to_numpy()

    val_indices = np.where(
        fold_array == fold
    )[0]

    train_indices = np.where(
        fold_array != fold
    )[0]

    # Compute angle statistics from training data only.
    angle_mean = np.nanmean(
        X_train_ang[
            train_indices,
            0
        ]
    )

    angle_std = np.nanstd(
        X_train_ang[
            train_indices,
            0
        ]
    )

    # Compute image statistics from training data only.
    train_images = X_train_img[
        train_indices
    ]

    image_mean = train_images.mean(
        axis=(0, 2, 3),
        dtype=np.float64
    ).astype(np.float32)

    image_std = train_images.std(
        axis=(0, 2, 3),
        dtype=np.float64
    ).astype(np.float32)

    train_transform = (
        build_train_transform(
            image_mean,
            image_std,
            augmentation,
        )
    )

    val_transform = (
        build_val_transform(
            image_mean,
            image_std,
        )
    )

    train_dataset = IcebergDataset(
        images=X_train_img,
        angles=X_train_ang,
        angle_missing=X_train_ang_missing,
        ids=train_ids,
        indices=train_indices,
        angle_mean=angle_mean,
        angle_std=angle_std,
        labels=Y_train,
        transform=train_transform,
    )

    val_dataset = IcebergDataset(
        images=X_train_img,
        angles=X_train_ang,
        angle_missing=X_train_ang_missing,
        ids=train_ids,
        indices=val_indices,
        angle_mean=angle_mean,
        angle_std=angle_std,
        labels=Y_train,
        transform=val_transform,
    )

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
        generator=generator,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(NUM_WORKERS > 0),
    )

    return (
        train_loader,
        val_loader
    )

# Part B — Multimodal EfficientNet Training Pipeline

In [8]:
class IcebergModel(nn.Module):
    def __init__(self, use_pretrained=True, dropout_p=0.4, use_angle=True):
        super().__init__()
        if not 0.0 <= dropout_p < 1.0:
            raise ValueError("dropout_p must be between 0 and 1")
        self.use_angle = use_angle
        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
            if use_pretrained else None
        )
        self.image_tower = models.efficientnet_b0(weights=weights)
        self.image_tower.classifier = nn.Identity()
        self.angle_tower = nn.Sequential(nn.Linear(1, 16), nn.ReLU())
        self.fusion_head = nn.Sequential(
            nn.Linear(1296, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, image, angle):
        image = F.interpolate(
            image, size=(224, 224), mode="bilinear", align_corners=False
        )
        image_features = self.image_tower(image)
        if self.use_angle:
            angle_features = self.angle_tower(angle)
        else:
            # Keep fusion dimensions unchanged, but remove angle information.
            angle_features = torch.zeros(
                image_features.size(0), 16,
                device=image_features.device,
                dtype=image_features.dtype,
            )
        features = torch.cat([image_features, angle_features], dim=1)
        return self.fusion_head(features)


In [9]:
## old model used for ablation study
class IcebergResNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.backbone = models.resnet18(
            weights=None
        )

        self.backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.backbone.maxpool = nn.Identity()

        num_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Linear(
            num_features,
            1
        )

        self.sigmoid = nn.Sigmoid()


    def forward(
        self,
        image,
        angle=None
    ):

        prediction = self.backbone(
            image
        )

        prediction = self.sigmoid(
            prediction
        )

        return prediction

In [10]:
def create_model(model_name, dropout_p=0.4, use_angle=True):
    if model_name == "efficientnet":
        return IcebergModel(
            use_pretrained=USE_PRETRAINED,
            dropout_p=dropout_p,
            use_angle=use_angle,
        )
    if model_name == "resnet":
        return IcebergResNet()
    raise ValueError(f"Unknown model: {model_name}")


In [11]:
def train_one_epoch(
    model, train_loader, criterion, optimizer, device, label_smoothing=0.0
):
    if not 0.0 <= label_smoothing < 1.0:
        raise ValueError("label_smoothing must be in [0, 1)")
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device, non_blocking=True)
        angles = batch["angle"].to(device, non_blocking=True)
        targets = batch["target"].unsqueeze(1).to(device, non_blocking=True)

        # Binary label smoothing: 0 -> eps/2, 1 -> 1 - eps/2.
        training_targets = targets * (1.0 - label_smoothing) + 0.5 * label_smoothing

        optimizer.zero_grad()
        predictions = model(images, angles)
        loss = criterion(predictions, training_targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    return running_loss / len(train_loader.dataset)


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device, non_blocking=True)
            angles = batch["angle"].to(device, non_blocking=True)
            targets = batch["target"].unsqueeze(1).to(device, non_blocking=True)
            predictions = model(images, angles)
            # Validation always uses original 0/1 labels.
            loss = criterion(predictions, targets)
            running_loss += loss.item() * images.size(0)
    return running_loss / len(val_loader.dataset)


In [12]:
set_seed(SEED)

check_train_loader, check_val_loader = (
    create_fold_loaders(
        fold=0,
        augmentation="baseline",
        batch_size=BATCH_SIZE,
        seed=SEED,
    )
)

check_model = IcebergModel(
    use_pretrained=USE_PRETRAINED
).to(device)

batch = next(
    iter(check_train_loader)
)

images = batch[
    "image"
].to(device)

angles = batch[
    "angle"
].to(device)

targets = batch[
    "target"
].unsqueeze(1).to(device)

check_model.eval()

with torch.no_grad():

    predictions = check_model(
        images,
        angles
    )

print(
    "Images:",
    images.shape
)

print(
    "Angles:",
    angles.shape
)

print(
    "Targets:",
    targets.shape
)

print(
    "Predictions:",
    predictions.shape
)

del check_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 174MB/s]


Images: torch.Size([32, 3, 75, 75])
Angles: torch.Size([32, 1])
Targets: torch.Size([32, 1])
Predictions: torch.Size([32, 1])


In [13]:
EXPERIMENT_CACHE = {}


def train_one_fold(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, max_epochs, patience,
    use_scheduler=True, use_early_stopping=True, label_smoothing=0.0,
):
    best_val_loss = float("inf")
    best_train_loss = None
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0
    final_train_loss = None
    final_val_loss = None
    final_epoch = 0

    for epoch in range(max_epochs):
        current_lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device,
            label_smoothing=label_smoothing,
        )
        val_loss = validate(model, val_loader, criterion, device)
        print(
            f"Epoch {epoch + 1:02d} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_loss = train_loss
            best_epoch = epoch + 1
            if use_early_stopping:
                best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        final_train_loss = train_loss
        final_val_loss = val_loss
        final_epoch = epoch + 1

        if use_early_stopping and epochs_without_improvement >= patience:
            print("Early stopping triggered.", flush=True)
            break
        if use_scheduler and scheduler is not None:
            scheduler.step()

    if use_early_stopping:
        selected_train_loss = best_train_loss
        selected_val_loss = best_val_loss
        selected_epoch = best_epoch
        selected_state = best_model_state
    else:
        selected_train_loss = final_train_loss
        selected_val_loss = final_val_loss
        selected_epoch = final_epoch
        selected_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(selected_state)
    return selected_train_loss, selected_val_loss, selected_epoch, selected_state


def run_5fold_experiment(
    augmentation="baseline", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    experiment_name=None, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    initial_lr=INITIAL_LR, min_lr=MIN_LR, patience=PATIENCE,
    optimizer_name="adam", weight_decay=0.0, dropout_p=0.4,
    label_smoothing=0.0, use_angle=True,
    save_models=False, force_run=False,
):
    if optimizer_name not in ("adam", "adamw"):
        raise ValueError("optimizer_name must be 'adam' or 'adamw'")
    if weight_decay < 0:
        raise ValueError("weight_decay must be nonnegative")
    if experiment_name is None:
        experiment_name = f"{model_name}_{augmentation}_{optimizer_name}"

    # Reuse identical experiments within the same notebook run.
    cache_key = (
        augmentation, model_name, use_scheduler, use_early_stopping,
        batch_size, max_epochs, initial_lr, min_lr, patience,
        optimizer_name, float(weight_decay), float(dropout_p),
        float(label_smoothing), bool(use_angle), bool(USE_PRETRAINED),
        NUM_FOLDS, SEED,
    )
    if cache_key in EXPERIMENT_CACHE and not save_models and not force_run:
        print(f"Reusing results: {experiment_name}")
        cached = EXPERIMENT_CACHE[cache_key].copy()
        cached["experiment"] = experiment_name
        cached.to_csv(WORK_DIR / f"{experiment_name}_results.csv", index=False)
        return cached

    print(
        f"\n=== {experiment_name} | {model_name} | {augmentation} | "
        f"{optimizer_name} | WD={weight_decay} | Dropout={dropout_p} | "
        f"Smooth={label_smoothing} | Angle={use_angle} | "
        f"Scheduler={use_scheduler} | Early={use_early_stopping} ===",
        flush=True,
    )
    fold_results = []
    for fold in range(NUM_FOLDS):
        print(f"\n========== Fold {fold} ==========", flush=True)
        fold_seed = SEED + fold
        set_seed(fold_seed)
        train_loader, val_loader = create_fold_loaders(
            fold=fold, augmentation=augmentation,
            batch_size=batch_size, seed=fold_seed,
        )
        model = create_model(
            model_name, dropout_p=dropout_p, use_angle=use_angle
        ).to(device)
        criterion = nn.BCELoss()
        optimizer_class = optim.Adam if optimizer_name == "adam" else optim.AdamW
        optimizer = optimizer_class(
            model.parameters(), lr=initial_lr, weight_decay=weight_decay,
        )
        scheduler = (
            optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max_epochs, eta_min=min_lr
            ) if use_scheduler else None
        )

        selected_train_loss, selected_val_loss, selected_epoch, selected_state = train_one_fold(
            model=model, train_loader=train_loader, val_loader=val_loader,
            criterion=criterion, optimizer=optimizer, scheduler=scheduler,
            device=device, max_epochs=max_epochs, patience=patience,
            use_scheduler=use_scheduler, use_early_stopping=use_early_stopping,
            label_smoothing=label_smoothing,
        )

        if save_models:
            cpu_state = {k: v.detach().cpu() for k, v in selected_state.items()}
            torch.save(cpu_state, WORK_DIR / f"{experiment_name}_fold_{fold}.pth")

        fold_results.append({
            "experiment": experiment_name,
            "fold": fold,
            "model": model_name,
            "augmentation": augmentation,
            "optimizer": optimizer_name,
            "weight_decay": weight_decay,
            "dropout": dropout_p,
            "label_smoothing": label_smoothing,
            "use_angle": use_angle,
            "scheduler": use_scheduler,
            "early_stopping": use_early_stopping,
            "selected_epoch": selected_epoch,
            "train_loss": selected_train_loss,
            "val_loss": selected_val_loss,
        })
        print(
            f"Fold {fold}: selected epoch={selected_epoch}, "
            f"train={selected_train_loss:.4f}, val={selected_val_loss:.4f}",
            flush=True,
        )
        del model, train_loader, val_loader, optimizer, scheduler, selected_state
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results = pd.DataFrame(fold_results)
    results.to_csv(WORK_DIR / f"{experiment_name}_results.csv", index=False)
    EXPERIMENT_CACHE[cache_key] = results.copy()
    print(
        f"Mean 5-Fold Train Loss: {results['train_loss'].mean():.4f} "
        f"(std={results['train_loss'].std():.4f}) | "
        f"Mean 5-Fold Val Loss: {results['val_loss'].mean():.4f} "
        f"(std={results['val_loss'].std():.4f})",
        flush=True,
    )
    return results


In [14]:
def make_summary(experiment_results, label_col="Experiment"):
    rows = []
    for name, results in experiment_results.items():
        if len(results) != NUM_FOLDS:
            raise ValueError(f"{name}: expected {NUM_FOLDS} folds, got {len(results)}")
        rows.append({
            label_col: name,
            "Mean Train Loss": results["train_loss"].mean(),
            "Std Train Loss": results["train_loss"].std(),
            "Mean CV Loss": results["val_loss"].mean(),
            "Std CV Loss": results["val_loss"].std(),
        })
    return pd.DataFrame(rows).sort_values("Mean CV Loss").reset_index(drop=True)


def mean_cv_loss(results):
    return float(results["val_loss"].mean())



## Local-only CSV Restore — Current Notebook

Restore completed experiments only from:
`/kaggle/working/iceberg-data/`

Rules:
- Never search `/kaggle/input`.
- Never reuse CSV files belonging to other experiment names.
- Verify five folds and recorded hyperparameters.
- If results are missing or incompatible, stop without training when RESTORE_ONLY=True.


In [15]:

# ============================================================
# LOCAL-ONLY CSV RESTORE
# Replace the entire old Auto-restore code cell with this cell.
# ============================================================

import inspect
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# True: Only restore saved results; never start model training.
RESTORE_ONLY = False

# Existing CSVs may have no matching _config.json.
# True: allow legacy CSVs if recorded fields match and
#       unrecorded settings equal the original defaults.
# False: reject all CSVs without complete config metadata.
ALLOW_LEGACY_LOCAL_CSV = True

# Only use this notebook's local working directory.
LOCAL_RESULTS = Path("/kaggle/working/iceberg-data")

if WORK_DIR.resolve() != LOCAL_RESULTS.resolve():
    raise RuntimeError("WORK_DIR does not match LOCAL_RESULTS")


# ------------------------------------------------------------
# Preserve the original training function.
# ------------------------------------------------------------

_current_runner = run_5fold_experiment

if hasattr(_current_runner, "_local_only_raw_runner"):
    _raw_runner = _current_runner._local_only_raw_runner

elif (
    hasattr(_current_runner, "__signature__")
    and "_ORIGINAL_ICEBERG_RUNNER" in globals()
):
    _raw_runner = _ORIGINAL_ICEBERG_RUNNER

else:
    _raw_runner = _current_runner

_raw_signature = inspect.signature(_raw_runner)


# ------------------------------------------------------------
# Original defaults for legacy CSVs.
# ------------------------------------------------------------

_LEGACY_DEFAULTS = {
    "batch_size": 32,
    "max_epochs": 30,
    "initial_lr": 0.001,
    "min_lr": 0.00001,
    "patience": 8,
    "seed": 42,
    "num_folds": 5,
    "use_pretrained": True,
}


def _requested_config(p):
    keys = (
        "augmentation",
        "model_name",
        "use_scheduler",
        "use_early_stopping",
        "batch_size",
        "max_epochs",
        "initial_lr",
        "min_lr",
        "patience",
        "optimizer_name",
        "weight_decay",
        "dropout_p",
        "label_smoothing",
        "use_angle",
    )

    cfg = {key: p[key] for key in keys}

    cfg.update(
        seed=int(SEED),
        num_folds=int(NUM_FOLDS),
        use_pretrained=bool(USE_PRETRAINED),
    )

    return cfg


# ------------------------------------------------------------
# Verify a local CSV before restoring it.
# ------------------------------------------------------------

def _read_verified_local_csv(csv_path, p, name):

    try:
        df = pd.read_csv(csv_path)

        required = {
            "experiment",
            "fold",
            "train_loss",
            "val_loss",
            "selected_epoch",
            "model",
            "augmentation",
            "optimizer",
            "weight_decay",
            "dropout",
            "label_smoothing",
            "use_angle",
            "scheduler",
            "early_stopping",
        }

        if not required.issubset(df.columns):
            raise ValueError("Required result columns are missing")

        if len(df) != NUM_FOLDS:
            raise ValueError(
                f"Expected {NUM_FOLDS} rows, got {len(df)}"
            )

        folds = pd.to_numeric(
            df["fold"], errors="raise"
        ).to_numpy()

        if not np.array_equal(
            np.sort(folds),
            np.arange(NUM_FOLDS)
        ):
            raise ValueError("Fold IDs do not match")

        losses = df[
            ["train_loss", "val_loss"]
        ].to_numpy(dtype=float)

        if not np.isfinite(losses).all():
            raise ValueError("Invalid train/validation loss")

        epochs = pd.to_numeric(
            df["selected_epoch"], errors="raise"
        ).to_numpy()

        if (
            not np.isfinite(epochs).all()
            or (epochs < 1).any()
            or (epochs > p["max_epochs"]).any()
        ):
            raise ValueError("Invalid selected epoch")

        # Check experiment identity and discrete parameters.
        strings = {
            "experiment": name,
            "model": p["model_name"],
            "augmentation": p["augmentation"],
            "optimizer": p["optimizer_name"],
        }

        for column, value in strings.items():
            actual = df[column].astype(str).str.lower()

            if not actual.eq(str(value).lower()).all():
                raise ValueError(
                    f"Experiment mismatch: {column}"
                )

        # Check numerical hyperparameters.
        numeric = {
            "weight_decay": p["weight_decay"],
            "dropout": p["dropout_p"],
            "label_smoothing": p["label_smoothing"],
        }

        for column, value in numeric.items():
            actual = pd.to_numeric(
                df[column], errors="raise"
            ).to_numpy(dtype=float)

            if not np.isclose(
                actual,
                float(value),
                rtol=1e-9,
                atol=1e-12,
            ).all():
                raise ValueError(
                    f"Parameter mismatch: {column}"
                )

        # Check boolean parameters.
        flags = {
            "use_angle": p["use_angle"],
            "scheduler": p["use_scheduler"],
            "early_stopping": p["use_early_stopping"],
        }

        for column, value in flags.items():
            actual = df[column].astype(str).str.lower()

            if not actual.eq(
                str(bool(value)).lower()
            ).all():
                raise ValueError(
                    f"Parameter mismatch: {column}"
                )

        # Check full metadata when available.
        meta_path = csv_path.with_name(
            name + "_config.json"
        )

        current_config = _requested_config(p)

        if meta_path.exists():

            saved_config = json.loads(
                meta_path.read_text(encoding="utf-8")
            )

            if saved_config != current_config:
                raise ValueError(
                    "Full configuration JSON does not match"
                )

        else:

            if not ALLOW_LEGACY_LOCAL_CSV:
                raise ValueError(
                    "Legacy CSV lacks full config metadata"
                )

            for key, value in _LEGACY_DEFAULTS.items():

                if current_config[key] != value:
                    raise ValueError(
                        f"Legacy CSV cannot verify {key}"
                    )

            print(
                f"[LEGACY LOCAL CSV] {name}: "
                "unrecorded settings assumed to be "
                "original defaults."
            )

        return df.sort_values(
            "fold"
        ).reset_index(drop=True)

    except (
        OSError,
        ValueError,
        TypeError,
        KeyError,
    ) as exc:

        raise RuntimeError(
            f"Local CSV cannot be safely reused:\n"
            f"{csv_path}\n"
            f"Reason: {exc}"
        ) from exc


# ------------------------------------------------------------
# Replace the public training function with a safe wrapper.
# ------------------------------------------------------------

def run_5fold_experiment(*args, **kwargs):

    bound = _raw_signature.bind(*args, **kwargs)
    bound.apply_defaults()

    p = dict(bound.arguments)

    name = p["experiment_name"] or (
        f"{p['model_name']}_"
        f"{p['augmentation']}_"
        f"{p['optimizer_name']}"
    )

    if not re.fullmatch(r"[A-Za-z0-9_-]+", name):
        raise ValueError(
            f"Invalid experiment_name: {name!r}"
        )

    # ONLY inspect the exact local CSV filename.
    csv_path = LOCAL_RESULTS / f"{name}_results.csv"

    if csv_path.exists():

        if csv_path.is_symlink() or not csv_path.is_file():
            raise RuntimeError(
                f"Untrusted local CSV: {csv_path}"
            )

        if not p["force_run"] and not p["save_models"]:

            restored = _read_verified_local_csv(
                csv_path, p, name
            )

            print(
                f"[LOCAL RESTORED - NO TRAINING] "
                f"{csv_path.name}"
            )
            print(
                f"Mean CV Loss: "
                f"{restored['val_loss'].mean():.6f}"
            )

            return restored

        if RESTORE_ONLY:
            raise RuntimeError(
                f"Training blocked by RESTORE_ONLY: {name}"
            )

        # Do not silently overwrite previously saved results.
        raise RuntimeError(
            f"Local result already exists: {csv_path}"
        )

    # No CSV found: never train in restore-only mode.
    if RESTORE_ONLY:

        raise FileNotFoundError(
            f"[RESTORE_ONLY] Missing local CSV:\n"
            f"{csv_path}\n"
            "No training was started. "
            "No other folder was searched."
        )

    # Training is allowed only for a missing experiment.
    LOCAL_RESULTS.mkdir(
        parents=True,
        exist_ok=True
    )

    result = _raw_runner(*args, **kwargs)

    # Save complete settings for future strict verification.
    metadata_path = (
        LOCAL_RESULTS / f"{name}_config.json"
    )

    temporary = metadata_path.with_suffix(".tmp")

    with temporary.open(
        "w", encoding="utf-8"
    ) as f:

        json.dump(
            _requested_config(p),
            f,
            indent=2,
            ensure_ascii=False,
        )

    os.replace(temporary, metadata_path)

    return result


# Keep the original function signature available.
run_5fold_experiment.__signature__ = _raw_signature
run_5fold_experiment._local_only_raw_runner = _raw_runner

print("LOCAL-ONLY CSV restore enabled.")
print("Reading only:", LOCAL_RESULTS)
print("RESTORE_ONLY =", RESTORE_ONLY)
print("ALLOW_LEGACY_LOCAL_CSV =", ALLOW_LEGACY_LOCAL_CSV)


LOCAL-ONLY CSV restore enabled.
Reading only: /kaggle/working/iceberg-data
RESTORE_ONLY = False
ALLOW_LEGACY_LOCAL_CSV = True


# Version 0: Baseline model

In [16]:
# Report Baseline: ResNet-18 without training enhancements

report_baseline_results = run_5fold_experiment(
    augmentation="baseline",
    model_name="resnet",
    use_scheduler=False,
    use_early_stopping=False,
    optimizer_name="adam",
    weight_decay=0.0,
    dropout_p=0.0,
    label_smoothing=0.0,
    use_angle=False,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    initial_lr=INITIAL_LR,
    min_lr=MIN_LR,
    patience=PATIENCE,
    experiment_name="report_basic_resnet18_baseline",
)

print("=== Report Baseline ===")
display(make_summary({
    "Basic ResNet-18 Baseline": report_baseline_results
}))

[LEGACY LOCAL CSV] report_basic_resnet18_baseline: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] report_basic_resnet18_baseline_results.csv
Mean CV Loss: 0.447342
=== Report Baseline ===


,Experiment,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,Basic ResNet-18 Baseline,0.039068,0.035506,0.447342,0.103488


# Version 1 — Progressive Ablation
## Stage 1 — ResNet-18 Image-Only Baseline
For comparability, Adam (WD=0), the same scheduler, early stopping, folds and seeds are used.


In [17]:
old_model_results = run_5fold_experiment(
    augmentation="baseline", model_name="resnet",
    dropout_p=0.0, use_angle=False,
    experiment_name="v1_stage1_resnet_baseline",
)


[LEGACY LOCAL CSV] v1_stage1_resnet_baseline: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage1_resnet_baseline_results.csv
Mean CV Loss: 0.262480


## Version 1 · Stage 2 — Improved Reference Model
EfficientNet-B0 + Angle + Scheduler + Early Stopping, with Dropout disabled (`p=0`). Each variant changes one training-strategy factor.


In [18]:
full_results = run_5fold_experiment(
    augmentation="baseline", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_stage2_reference",
)


[LEGACY LOCAL CSV] v1_stage2_reference: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage2_reference_results.csv
Mean CV Loss: 0.260346


### Stage 2.1 — No Early Stopping


In [19]:
no_early_stopping_results = run_5fold_experiment(
    augmentation="baseline", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=False,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_no_early_stopping",
)


[LEGACY LOCAL CSV] v1_no_early_stopping: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_no_early_stopping_results.csv
Mean CV Loss: 0.460059


### Stage 2.2 — No Scheduler


In [20]:
no_scheduler_results = run_5fold_experiment(
    augmentation="baseline", model_name="efficientnet",
    use_scheduler=False, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_no_scheduler",
)


[LEGACY LOCAL CSV] v1_no_scheduler: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_no_scheduler_results.csv
Mean CV Loss: 0.260336


### Stage 2 — Model & Training Strategy Summary


In [21]:
stage2_results = {
    "ResNet-18 Baseline": old_model_results,
    "Improved Reference": full_results,
    "No Early Stopping": no_early_stopping_results,
    "No Scheduler": no_scheduler_results,
}
ablation_summary = make_summary(stage2_results)
display(ablation_summary)
ablation_summary.to_csv(WORK_DIR / "v1_stage2_summary.csv", index=False)


,Experiment,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,No Scheduler,0.184976,0.052066,0.260336,0.059380
1,Improved Reference,0.207336,0.016931,0.260346,0.062920
2,ResNet-18 Baseline,0.236438,0.093763,0.262480,0.041158
3,No Early Stopping,0.005010,0.002532,0.460059,0.079211


## Version 1 · Stage 3 — Augmentation Comparison
Only training images are augmented. All other settings match the Stage 2 Reference.


In [22]:
baseline_results = full_results.copy()


### Horizontal Flip


In [23]:
hflip_results = run_5fold_experiment(
    augmentation="hflip", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_flip_hflip",
)


[LEGACY LOCAL CSV] v1_flip_hflip: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_flip_hflip_results.csv
Mean CV Loss: 0.251187


### Vertical Flip


In [24]:
vflip_results = run_5fold_experiment(
    augmentation="vflip", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_flip_vflip",
)


[LEGACY LOCAL CSV] v1_flip_vflip: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_flip_vflip_results.csv
Mean CV Loss: 0.228076


### Horizontal + Vertical Flip


In [25]:
hvflip_results = run_5fold_experiment(
    augmentation="hvflip", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_flip_hvflip",
)


[LEGACY LOCAL CSV] v1_flip_hvflip: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_flip_hvflip_results.csv
Mean CV Loss: 0.223458


### Rotation


In [26]:
rotation_results = run_5fold_experiment(
    augmentation="rotation", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_rotation",
)


[LEGACY LOCAL CSV] v1_rotation: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_rotation_results.csv
Mean CV Loss: 0.202408


### Light Crop


In [27]:
crop_light_results = run_5fold_experiment(
    augmentation="crop_light", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_crop_light",
)


[LEGACY LOCAL CSV] v1_crop_light: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_crop_light_results.csv
Mean CV Loss: 0.213942


### Medium Crop


In [28]:
crop_medium_results = run_5fold_experiment(
    augmentation="crop_medium", model_name="efficientnet",
    use_scheduler=True, use_early_stopping=True,
    dropout_p=0.0, use_angle=True,
    experiment_name="v1_crop_medium",
)


[LEGACY LOCAL CSV] v1_crop_medium: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_crop_medium_results.csv
Mean CV Loss: 0.247361


### Stage 3 Summary — Select Best Augmentation


In [29]:
augmentation_results = {
    "baseline": baseline_results,
    "hflip": hflip_results,
    "vflip": vflip_results,
    "hvflip": hvflip_results,
    "rotation": rotation_results,
    "crop_light": crop_light_results,
    "crop_medium": crop_medium_results,
}
augmentation_summary = make_summary(augmentation_results, "Augmentation")
display(augmentation_summary)
augmentation_summary.to_csv(WORK_DIR / "v1_stage3_augmentation_summary.csv", index=False)

best_augmentation = str(augmentation_summary.iloc[0]["Augmentation"])
print("Best augmentation selected by Mean CV Loss:", best_augmentation)


,Augmentation,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,rotation,0.183117,0.069864,0.202408,0.053424
1,crop_light,0.213186,0.031764,0.213942,0.042104
2,hvflip,0.166411,0.016593,0.223458,0.055679
3,vflip,0.162139,0.058011,0.228076,0.060389
4,crop_medium,0.190908,0.046484,0.247361,0.039496
5,hflip,0.147348,0.051406,0.251187,0.042540
6,baseline,0.207336,0.016931,0.260346,0.062920


Best augmentation selected by Mean CV Loss: rotation


## Version 1 · Stage 4 — Optimizer × Weight Decay × Label Smoothing × Dropout
Complete 2 × 2 × 2 × 2 factorial design (16 configurations). A reuses the best augmentation from Stage 3 with Dropout 0. All experiments use that same augmentation, EfficientNet-B0 + Angle, Scheduler, Early Stopping and five folds.


In [30]:
# A is identical to the winning Stage 3 augmentation experiment.
stage4_base_results = augmentation_results[best_augmentation].copy()


### B — Adam + Weight Decay 1e-4 (Dropout 0)


In [31]:
stage4_adam_wd_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=1e-4,
    dropout_p=0.0, label_smoothing=0.0,
    experiment_name="v1_stage4_B_adam_wd",
)


[LEGACY LOCAL CSV] v1_stage4_B_adam_wd: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_B_adam_wd_results.csv
Mean CV Loss: 0.206801


### C — AdamW + No Weight Decay (Dropout 0)


In [32]:
stage4_adamw_no_wd_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0,
    dropout_p=0.0, label_smoothing=0.0,
    experiment_name="v1_stage4_C_adamw_no_wd",
)


[LEGACY LOCAL CSV] v1_stage4_C_adamw_no_wd: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_C_adamw_no_wd_results.csv
Mean CV Loss: 0.202408


### D — AdamW + Weight Decay 1e-4 (Dropout 0)


In [33]:
stage4_adamw_wd_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=1e-4,
    dropout_p=0.0, label_smoothing=0.0,
    experiment_name="v1_stage4_D_adamw_wd",
)


[LEGACY LOCAL CSV] v1_stage4_D_adamw_wd: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_D_adamw_wd_results.csv
Mean CV Loss: 0.208066


### E — Adam + Label Smoothing 0.1 (Dropout 0)


In [34]:
stage4_smoothing_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0,
    dropout_p=0.0, label_smoothing=0.1,
    experiment_name="v1_stage4_E_label_smoothing",
)


[LEGACY LOCAL CSV] v1_stage4_E_label_smoothing: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_E_label_smoothing_results.csv
Mean CV Loss: 0.223711


### F — Adam, WD=0.0001, Smoothing=0.1, Dropout=0

In [35]:
stage4_adam_wd_smoothing_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0001,
    dropout_p=0.0, label_smoothing=0.1,
    experiment_name="v1_stage4_F_adam_wd_smoothing",
)

[LEGACY LOCAL CSV] v1_stage4_F_adam_wd_smoothing: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_F_adam_wd_smoothing_results.csv
Mean CV Loss: 0.222392


### G — AdamW, WD=0, Smoothing=0.1, Dropout=0

In [36]:
stage4_adamw_smoothing_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0,
    dropout_p=0.0, label_smoothing=0.1,
    experiment_name="v1_stage4_G_adamw_smoothing",
)

[LEGACY LOCAL CSV] v1_stage4_G_adamw_smoothing: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_G_adamw_smoothing_results.csv
Mean CV Loss: 0.223711


### H — AdamW, WD=0.0001, Smoothing=0.1, Dropout=0

In [37]:
stage4_adamw_wd_smoothing_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0001,
    dropout_p=0.0, label_smoothing=0.1,
    experiment_name="v1_stage4_H_adamw_wd_smoothing",
)

[LEGACY LOCAL CSV] v1_stage4_H_adamw_wd_smoothing: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_H_adamw_wd_smoothing_results.csv
Mean CV Loss: 0.222411


### I — Adam, WD=0, Smoothing=0, Dropout=0.4

In [38]:
stage4_adam_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0,
    dropout_p=0.4, label_smoothing=0.0,
    experiment_name="v1_stage4_I_adam_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_I_adam_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_I_adam_dropout_results.csv
Mean CV Loss: 0.216965


### J — Adam, WD=0.0001, Smoothing=0, Dropout=0.4

In [39]:
stage4_adam_wd_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0001,
    dropout_p=0.4, label_smoothing=0.0,
    experiment_name="v1_stage4_J_adam_wd_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_J_adam_wd_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_J_adam_wd_dropout_results.csv
Mean CV Loss: 0.207749


### K — AdamW, WD=0, Smoothing=0, Dropout=0.4

In [40]:
stage4_adamw_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0,
    dropout_p=0.4, label_smoothing=0.0,
    experiment_name="v1_stage4_K_adamw_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_K_adamw_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_K_adamw_dropout_results.csv
Mean CV Loss: 0.216965


### L — AdamW, WD=0.0001, Smoothing=0, Dropout=0.4

In [41]:
stage4_adamw_wd_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0001,
    dropout_p=0.4, label_smoothing=0.0,
    experiment_name="v1_stage4_L_adamw_wd_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_L_adamw_wd_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_L_adamw_wd_dropout_results.csv
Mean CV Loss: 0.209560


### M — Adam, WD=0, Smoothing=0.1, Dropout=0.4

In [42]:
stage4_adam_smoothing_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0,
    dropout_p=0.4, label_smoothing=0.1,
    experiment_name="v1_stage4_M_adam_smoothing_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_M_adam_smoothing_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_M_adam_smoothing_dropout_results.csv
Mean CV Loss: 0.218310


### N — Adam, WD=0.0001, Smoothing=0.1, Dropout=0.4

In [43]:
stage4_adam_wd_smoothing_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adam", weight_decay=0.0001,
    dropout_p=0.4, label_smoothing=0.1,
    experiment_name="v1_stage4_N_adam_wd_smoothing_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_N_adam_wd_smoothing_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_N_adam_wd_smoothing_dropout_results.csv
Mean CV Loss: 0.219762


### O — AdamW, WD=0, Smoothing=0.1, Dropout=0.4

In [44]:
stage4_adamw_smoothing_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0,
    dropout_p=0.4, label_smoothing=0.1,
    experiment_name="v1_stage4_O_adamw_smoothing_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_O_adamw_smoothing_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_O_adamw_smoothing_dropout_results.csv
Mean CV Loss: 0.218310


### P — AdamW, WD=0.0001, Smoothing=0.1, Dropout=0.4

In [45]:
stage4_adamw_wd_smoothing_dropout_results = run_5fold_experiment(
    augmentation=best_augmentation,
    model_name="efficientnet",
    optimizer_name="adamw", weight_decay=0.0001,
    dropout_p=0.4, label_smoothing=0.1,
    experiment_name="v1_stage4_P_adamw_wd_smoothing_dropout",
)

[LEGACY LOCAL CSV] v1_stage4_P_adamw_wd_smoothing_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] v1_stage4_P_adamw_wd_smoothing_dropout_results.csv
Mean CV Loss: 0.225681


### Stage 4 Summary — All 16 Combinations


In [46]:
# Version 1 Stage 4: 16-factor experiment results
stage4_experiments = [
    ('A. Base Model', stage4_base_results, 'Adam', 0.0, 0.0, 0.0),
    ('B. Adam + WD', stage4_adam_wd_results, 'Adam', 0.0001, 0.0, 0.0),
    ('C. AdamW No WD', stage4_adamw_no_wd_results, 'AdamW', 0.0, 0.0, 0.0),
    ('D. AdamW + WD', stage4_adamw_wd_results, 'AdamW', 0.0001, 0.0, 0.0),
    ('E. Adam + Smoothing', stage4_smoothing_results, 'Adam', 0.0, 0.0, 0.1),
    ('F. Adam + WD + Smoothing', stage4_adam_wd_smoothing_results, 'Adam', 0.0001, 0.0, 0.1),
    ('G. AdamW + Smoothing', stage4_adamw_smoothing_results, 'AdamW', 0.0, 0.0, 0.1),
    ('H. AdamW + WD + Smoothing', stage4_adamw_wd_smoothing_results, 'AdamW', 0.0001, 0.0, 0.1),
    ('I. Adam + Dropout', stage4_adam_dropout_results, 'Adam', 0.0, 0.4, 0.0),
    ('J. Adam + WD + Dropout', stage4_adam_wd_dropout_results, 'Adam', 0.0001, 0.4, 0.0),
    ('K. AdamW + Dropout', stage4_adamw_dropout_results, 'AdamW', 0.0, 0.4, 0.0),
    ('L. AdamW + WD + Dropout', stage4_adamw_wd_dropout_results, 'AdamW', 0.0001, 0.4, 0.0),
    ('M. Adam + Smoothing + Dropout', stage4_adam_smoothing_dropout_results, 'Adam', 0.0, 0.4, 0.1),
    ('N. Adam + WD + Smoothing + Dropout', stage4_adam_wd_smoothing_dropout_results, 'Adam', 0.0001, 0.4, 0.1),
    ('O. AdamW + Smoothing + Dropout', stage4_adamw_smoothing_dropout_results, 'AdamW', 0.0, 0.4, 0.1),
    ('P. AdamW + WD + Smoothing + Dropout', stage4_adamw_wd_smoothing_dropout_results, 'AdamW', 0.0001, 0.4, 0.1),
]

stage4_results = {}
stage4_setting_rows = []
for name, results, optimizer_name, wd, dropout_p, smoothing in stage4_experiments:
    assert len(results) == NUM_FOLDS, f"{name}: incorrect fold count"
    assert set(results["fold"]) == set(range(NUM_FOLDS)), f"{name}: incorrect fold IDs"
    assert (results["optimizer"] == optimizer_name.lower()).all(), f"{name}: optimizer mismatch"
    assert np.allclose(results["weight_decay"], wd), f"{name}: WD mismatch"
    assert np.allclose(results["dropout"], dropout_p), f"{name}: dropout mismatch"
    assert np.allclose(results["label_smoothing"], smoothing), f"{name}: smoothing mismatch"
    assert (results["augmentation"] == best_augmentation).all(), f"{name}: augmentation mismatch"
    assert results["use_angle"].all(), f"{name}: angle mismatch"
    assert results["scheduler"].all(), f"{name}: scheduler mismatch"
    assert results["early_stopping"].all(), f"{name}: early stopping mismatch"
    stage4_results[name] = results
    stage4_setting_rows.append({
        "Experiment": name,
        "Optimizer": optimizer_name,
        "WD": wd,
        "Dropout": dropout_p,
        "Smoothing": smoothing,
    })

stage4_summary = make_summary(stage4_results)
stage4_settings = pd.DataFrame(stage4_setting_rows)
stage4_summary = stage4_summary.merge(stage4_settings, on="Experiment")
stage4_summary = stage4_summary[[
    "Experiment", "Optimizer", "WD", "Dropout", "Smoothing",
    "Mean Train Loss", "Std Train Loss", "Mean CV Loss", "Std CV Loss",
]]
display(stage4_summary)
stage4_summary.to_csv(WORK_DIR / "v1_stage4_summary.csv", index=False)

best_stage4 = stage4_summary.iloc[0]
print("Best Stage 4 configuration:")
print(best_stage4[["Experiment", "Optimizer", "WD", "Dropout", "Smoothing", "Mean CV Loss"]].to_string())

,Experiment,Optimizer,WD,Dropout,Smoothing,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,A. Base Model,Adam,0.0000,0.0,0.0,0.183117,0.069864,0.202408,0.053424
1,C. AdamW No WD,AdamW,0.0000,0.0,0.0,0.183117,0.069864,0.202408,0.053424
2,B. Adam + WD,Adam,0.0001,0.0,0.0,0.204919,0.045626,0.206801,0.036108
3,J. Adam + WD + Dropout,Adam,0.0001,0.4,0.0,0.171995,0.046854,0.207749,0.038602
4,D. AdamW + WD,AdamW,0.0001,0.0,0.0,0.169392,0.056425,0.208066,0.043222
5,L. AdamW + WD + Dropout,AdamW,0.0001,0.4,0.0,0.206990,0.043306,0.209560,0.031928
6,K. AdamW + Dropout,AdamW,0.0000,0.4,0.0,0.174263,0.060680,0.216965,0.045074
7,I. Adam + Dropout,Adam,0.0000,0.4,0.0,0.174263,0.060680,0.216965,0.045074
8,M. Adam + Smoothing + Dropout,Adam,0.0000,0.4,0.1,0.277853,0.025887,0.218310,0.023153
9,O. AdamW + Smoothing + Dropout,AdamW,0.0000,0.4,0.1,0.277853,0.025887,0.218310,0.023153


Best Stage 4 configuration:
Experiment      A. Base Model
Optimizer                Adam
WD                        0.0
Dropout                   0.0
Smoothing                 0.0
Mean CV Loss         0.202408


### Cell 1 — Crop Light × Weight Decay

In [47]:
# Cross Experiment 1: Crop Light + Weight Decay

cross_crop_wd_results = run_5fold_experiment(
    augmentation="crop_light",
    model_name="efficientnet",
    optimizer_name="adam",
    weight_decay=1e-4,
    dropout_p=0.0,
    label_smoothing=0.0,
    use_angle=True,
    use_scheduler=True,
    use_early_stopping=True,
    experiment_name="cross_crop_light_wd",
)

[LEGACY LOCAL CSV] cross_crop_light_wd: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_crop_light_wd_results.csv
Mean CV Loss: 0.222092


### Cell 2 — Crop Light × Dropout

In [48]:
# Cross Experiment 2: Crop Light + Dropout

cross_crop_dropout_results = run_5fold_experiment(
    augmentation="crop_light",
    model_name="efficientnet",
    optimizer_name="adam",
    weight_decay=0.0,
    dropout_p=0.4,
    label_smoothing=0.0,
    use_angle=True,
    use_scheduler=True,
    use_early_stopping=True,
    experiment_name="cross_crop_light_dropout",
)

[LEGACY LOCAL CSV] cross_crop_light_dropout: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_crop_light_dropout_results.csv
Mean CV Loss: 0.223532


### Cell 3 — Rotation × Scheduler

In [49]:
# Cross Experiment 3: Rotation without Scheduler

cross_rotation_no_scheduler_results = run_5fold_experiment(
    augmentation="rotation",
    model_name="efficientnet",
    optimizer_name="adam",
    weight_decay=0.0,
    dropout_p=0.0,
    label_smoothing=0.0,
    use_angle=True,
    use_scheduler=False,
    use_early_stopping=True,
    experiment_name="cross_rotation_no_scheduler",
)

[LEGACY LOCAL CSV] cross_rotation_no_scheduler: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_rotation_no_scheduler_results.csv
Mean CV Loss: 0.212584


### Cell4 — Scheduler × Weight Decay

In [50]:
# Cross Experiment 4: Scheduler x Weight Decay
# Rotation, Adam, WD=1e-4, Dropout=0, Smoothing=0
# Only Scheduler is disabled.

cross_rotation_wd_no_scheduler_results = run_5fold_experiment(
    augmentation="rotation",
    model_name="efficientnet",
    optimizer_name="adam",
    weight_decay=1e-4,
    dropout_p=0.0,
    label_smoothing=0.0,
    use_angle=True,
    use_scheduler=False,
    use_early_stopping=True,
    experiment_name="cross_rotation_wd_no_scheduler",
)

[LEGACY LOCAL CSV] cross_rotation_wd_no_scheduler: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_rotation_wd_no_scheduler_results.csv
Mean CV Loss: 0.212759


### Cell 5 — summary

In [51]:
# Cell 5: Cross Experiment Summary

experiment_specs = [
    (
        "Rotation, WD=0, Scheduler=On",
        stage4_base_results,
        "rotation", 0.0, 0.0, True,
    ),
    (
        "Rotation, WD=1e-4, Scheduler=On",
        stage4_adam_wd_results,
        "rotation", 1e-4, 0.0, True,
    ),
    (
        "Crop Light, WD=0, Scheduler=On",
        crop_light_results,
        "crop_light", 0.0, 0.0, True,
    ),
    (
        "Crop Light, WD=1e-4, Scheduler=On",
        cross_crop_wd_results,
        "crop_light", 1e-4, 0.0, True,
    ),
    (
        "Rotation, Dropout=0.4",
        stage4_adam_dropout_results,
        "rotation", 0.0, 0.4, True,
    ),
    (
        "Crop Light, Dropout=0.4",
        cross_crop_dropout_results,
        "crop_light", 0.0, 0.4, True,
    ),
    (
        "Baseline, Scheduler=On",
        full_results,
        "baseline", 0.0, 0.0, True,
    ),
    (
        "Baseline, Scheduler=Off",
        no_scheduler_results,
        "baseline", 0.0, 0.0, False,
    ),
    (
        "Rotation, WD=0, Scheduler=Off",
        cross_rotation_no_scheduler_results,
        "rotation", 0.0, 0.0, False,
    ),
    (
        "Rotation, WD=1e-4, Scheduler=Off",
        cross_rotation_wd_no_scheduler_results,
        "rotation", 1e-4, 0.0, False,
    ),
]

cross_experiment_results = {}
settings_rows = []

for name, df, aug, wd, dropout, scheduler in experiment_specs:

    # Verify five-fold results
    assert len(df) == NUM_FOLDS, (
        f"{name}: incorrect number of folds"
    )

    assert set(df["fold"]) == set(range(NUM_FOLDS)), (
        f"{name}: missing or duplicate folds"
    )

    # Verify experimental settings
    assert (df["model"] == "efficientnet").all()
    assert (df["augmentation"] == aug).all()
    assert (df["optimizer"] == "adam").all()

    assert np.allclose(df["weight_decay"], wd)
    assert np.allclose(df["dropout"], dropout)
    assert np.allclose(df["label_smoothing"], 0.0)

    assert (df["use_angle"] == True).all()
    assert (df["scheduler"] == scheduler).all()
    assert (df["early_stopping"] == True).all()

    assert np.isfinite(
        df[["train_loss", "val_loss"]].to_numpy()
    ).all()

    cross_experiment_results[name] = df

    settings_rows.append({
        "Experiment": name,
        "Augmentation": aug,
        "WD": wd,
        "Dropout": dropout,
        "Scheduler": scheduler,
    })

print("All experiment settings verified successfully.")

# Calculate CV summary
cross_summary = make_summary(
    cross_experiment_results,
    label_col="Experiment",
)

settings_df = pd.DataFrame(settings_rows)

cross_summary = cross_summary.merge(
    settings_df,
    on="Experiment",
)

cross_summary = cross_summary[
    [
        "Experiment",
        "Augmentation",
        "WD",
        "Dropout",
        "Scheduler",
        "Mean Train Loss",
        "Std Train Loss",
        "Mean CV Loss",
        "Std CV Loss",
    ]
]

display(cross_summary.round(6))

# Save summary
cross_summary.to_csv(
    WORK_DIR / "cross_experiment_summary.csv",
    index=False,
)

print("Cross experiment summary saved.")


# Cell 6: Four Interaction Effects

def fold_val_loss(results):
    return (
        results
        .set_index("fold")["val_loss"]
        .sort_index()
        .astype(float)
    )


def paired_effect(reference, changed):
    ref = fold_val_loss(reference)
    new = fold_val_loss(changed)

    assert ref.index.equals(new.index)

    return new - ref


# 1. Augmentation x Weight Decay
# Effect = WD=1e-4 minus WD=0

wd_effect_rotation = paired_effect(
    stage4_base_results,
    stage4_adam_wd_results,
)

wd_effect_crop = paired_effect(
    crop_light_results,
    cross_crop_wd_results,
)


# 2. Augmentation x Dropout
# Effect = Dropout=0.4 minus Dropout=0

dropout_effect_rotation = paired_effect(
    stage4_base_results,
    stage4_adam_dropout_results,
)

dropout_effect_crop = paired_effect(
    crop_light_results,
    cross_crop_dropout_results,
)


# 3. Augmentation x Scheduler
# Effect = Scheduler=False minus Scheduler=True

scheduler_effect_baseline = paired_effect(
    full_results,
    no_scheduler_results,
)

scheduler_effect_rotation = paired_effect(
    stage4_base_results,
    cross_rotation_no_scheduler_results,
)


# 4. Scheduler x Weight Decay
# Effect = WD=1e-4 minus WD=0

wd_effect_scheduler_on = paired_effect(
    stage4_base_results,
    stage4_adam_wd_results,
)

wd_effect_scheduler_off = paired_effect(
    cross_rotation_no_scheduler_results,
    cross_rotation_wd_no_scheduler_results,
)


# Collect all four interactions

interaction_specs = [
    (
        "Augmentation x Weight Decay",
        "Rotation",
        "Crop Light",
        wd_effect_rotation,
        wd_effect_crop,
    ),
    (
        "Augmentation x Dropout",
        "Rotation",
        "Crop Light",
        dropout_effect_rotation,
        dropout_effect_crop,
    ),
    (
        "Augmentation x Scheduler",
        "Baseline",
        "Rotation",
        scheduler_effect_baseline,
        scheduler_effect_rotation,
    ),
    (
        "Scheduler x Weight Decay",
        "Scheduler On",
        "Scheduler Off",
        wd_effect_scheduler_on,
        wd_effect_scheduler_off,
    ),
]


summary_rows = []
detail_rows = []

for name, condition_1, condition_2, effect_1, effect_2 in interaction_specs:

    interaction = effect_2 - effect_1

    summary_rows.append({
        "Interaction": name,
        "Condition 1": condition_1,
        "Effect 1": effect_1.mean(),
        "Condition 2": condition_2,
        "Effect 2": effect_2.mean(),
        "Interaction Estimate": interaction.mean(),
        "Fold Std": interaction.std(),
    })

    for fold in interaction.index:

        detail_rows.append({
            "Interaction": name,
            "Fold": fold,
            "Effect 1": effect_1.loc[fold],
            "Effect 2": effect_2.loc[fold],
            "Interaction Estimate": interaction.loc[fold],
        })


interaction_summary = pd.DataFrame(summary_rows)
interaction_fold_details = pd.DataFrame(detail_rows)


print("=== Interaction Summary ===")
display(interaction_summary.round(6))

print("\n=== Per-Fold Interaction Details ===")
display(interaction_fold_details.round(6))


# Save both tables
interaction_summary.to_csv(
    WORK_DIR / "cross_interaction_summary.csv",
    index=False,
)

interaction_fold_details.to_csv(
    WORK_DIR / "cross_interaction_fold_details.csv",
    index=False,
)

print("Interaction analysis completed successfully.")

All experiment settings verified successfully.


,Experiment,Augmentation,WD,Dropout,Scheduler,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,"Rotation, WD=0, Scheduler=On",rotation,0.0000,0.0,True,0.183117,0.069864,0.202408,0.053424
1,"Rotation, WD=1e-4, Scheduler=On",rotation,0.0001,0.0,True,0.204919,0.045626,0.206801,0.036108
2,"Rotation, WD=0, Scheduler=Off",rotation,0.0000,0.0,False,0.182455,0.037834,0.212584,0.043475
3,"Rotation, WD=1e-4, Scheduler=Off",rotation,0.0001,0.0,False,0.200961,0.017730,0.212759,0.044189
4,"Crop Light, WD=0, Scheduler=On",crop_light,0.0000,0.0,True,0.213186,0.031764,0.213942,0.042104
5,"Rotation, Dropout=0.4",rotation,0.0000,0.4,True,0.174263,0.060680,0.216965,0.045074
6,"Crop Light, WD=1e-4, Scheduler=On",crop_light,0.0001,0.0,True,0.171216,0.050484,0.222092,0.035822
7,"Crop Light, Dropout=0.4",crop_light,0.0000,0.4,True,0.183375,0.026464,0.223532,0.036357
8,"Baseline, Scheduler=Off",baseline,0.0000,0.0,False,0.184976,0.052066,0.260336,0.059380
9,"Baseline, Scheduler=On",baseline,0.0000,0.0,True,0.207336,0.016931,0.260346,0.062920


Cross experiment summary saved.
=== Interaction Summary ===


,Interaction,Condition 1,Effect 1,Condition 2,Effect 2,Interaction Estimate,Fold Std
0,Augmentation x Weight Decay,Rotation,0.004392,Crop Light,0.008150,0.003758,0.015773
1,Augmentation x Dropout,Rotation,0.014557,Crop Light,0.009590,-0.004967,0.031747
2,Augmentation x Scheduler,Baseline,-0.000011,Rotation,0.010176,0.010186,0.006843
3,Scheduler x Weight Decay,Scheduler On,0.004392,Scheduler Off,0.000176,-0.004217,0.022071



=== Per-Fold Interaction Details ===


,Interaction,Fold,Effect 1,Effect 2,Interaction Estimate
0,Augmentation x Weight Decay,0,0.018577,0.023292,0.004715
1,Augmentation x Weight Decay,1,0.018308,-0.002228,-0.020536
2,Augmentation x Weight Decay,2,-0.030711,-0.007543,0.023168
3,Augmentation x Weight Decay,3,0.006764,0.009353,0.002589
4,Augmentation x Weight Decay,4,0.009023,0.017877,0.008854
5,Augmentation x Dropout,0,0.020160,0.024549,0.004389
6,Augmentation x Dropout,1,0.044002,-0.004530,-0.048532
7,Augmentation x Dropout,2,0.001794,-0.007012,-0.008806
8,Augmentation x Dropout,3,0.020704,0.009192,-0.011512
9,Augmentation x Dropout,4,-0.013875,0.025750,0.039625


Interaction analysis completed successfully.


## Version 1 - Stage 5

In [52]:
# Cross-stage experiments: six missing configurations

assert best_augmentation == "rotation", (
    "Stage 3 selected augmentation was not rotation."
)

cross_specs = [
    # ID, Augmentation, Scheduler, Weight Decay, Dropout
    ("C1", "crop_light", True,  1e-4, 0.0),
    ("C2", "crop_light", True,  0.0,  0.4),
    ("C3", "rotation",   False, 0.0,  0.0),
    ("C4", "rotation",   False, 1e-4, 0.0),
    ("C5", "rotation",   False, 0.0,  0.4),
    ("C6", "crop_light", False, 0.0,  0.0),
]

cross_new_results = {}

for tag, aug, scheduler_on, wd, dropout in cross_specs:

    print(
        f"\n===== {tag}: augmentation={aug}, "
        f"scheduler={scheduler_on}, "
        f"WD={wd}, dropout={dropout} ====="
    )

    cross_new_results[tag] = run_5fold_experiment(
        augmentation=aug,
        model_name="efficientnet",
        optimizer_name="adam",
        weight_decay=wd,
        dropout_p=dropout,
        label_smoothing=0.0,
        use_angle=True,
        use_scheduler=scheduler_on,
        use_early_stopping=True,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        experiment_name=(
            f"cross_{tag.lower()}_{aug}_"
            f"{'sched_on' if scheduler_on else 'sched_off'}"
        ),
    )

    df = cross_new_results[tag]

    print(
        f"{tag} Mean CV Loss: "
        f"{df['val_loss'].mean():.6f}"
    )


===== C1: augmentation=crop_light, scheduler=True, WD=0.0001, dropout=0.0 =====
[LEGACY LOCAL CSV] cross_c1_crop_light_sched_on: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_c1_crop_light_sched_on_results.csv
Mean CV Loss: 0.222092
C1 Mean CV Loss: 0.222092

===== C2: augmentation=crop_light, scheduler=True, WD=0.0, dropout=0.4 =====
[LEGACY LOCAL CSV] cross_c2_crop_light_sched_on: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_c2_crop_light_sched_on_results.csv
Mean CV Loss: 0.223532
C2 Mean CV Loss: 0.223532

===== C3: augmentation=rotation, scheduler=False, WD=0.0, dropout=0.0 =====
[LEGACY LOCAL CSV] cross_c3_rotation_sched_off: unrecorded settings assumed to be original defaults.
[LOCAL RESTORED - NO TRAINING] cross_c3_rotation_sched_off_results.csv
Mean CV Loss: 0.212584
C3 Mean CV Loss: 0.212584

===== C4: augmentation=rotation, scheduler=False, WD=0.0001, dropout=0.0 =====
[LEGACY LOCAL CSV

In [53]:
# Summary: 4 existing + 6 new configurations

cross_configs = [
    # Key, Augmentation, Scheduler, WD, Dropout, Results

    ("R_ON_BASE", "rotation", True, 0.0, 0.0,
     stage4_base_results),

    ("R_ON_WD", "rotation", True, 1e-4, 0.0,
     stage4_adam_wd_results),

    ("R_ON_DROP", "rotation", True, 0.0, 0.4,
     stage4_adam_dropout_results),

    ("C_ON_BASE", "crop_light", True, 0.0, 0.0,
     crop_light_results),

    ("C_ON_WD", "crop_light", True, 1e-4, 0.0,
     cross_new_results["C1"]),

    ("C_ON_DROP", "crop_light", True, 0.0, 0.4,
     cross_new_results["C2"]),

    ("R_OFF_BASE", "rotation", False, 0.0, 0.0,
     cross_new_results["C3"]),

    ("R_OFF_WD", "rotation", False, 1e-4, 0.0,
     cross_new_results["C4"]),

    ("R_OFF_DROP", "rotation", False, 0.0, 0.4,
     cross_new_results["C5"]),

    ("C_OFF_BASE", "crop_light", False, 0.0, 0.0,
     cross_new_results["C6"]),
]

cross_all_results = {}
config_info = []

for key, aug, sched, wd, drop, df in cross_configs:

    assert len(df) == NUM_FOLDS
    assert set(df["fold"]) == set(range(NUM_FOLDS)), key

    assert (df["model"] == "efficientnet").all(), key
    assert (df["augmentation"] == aug).all(), key
    assert (df["optimizer"] == "adam").all(), key
    assert (df["scheduler"] == sched).all(), key
    assert (df["early_stopping"] == True).all(), key
    assert (df["use_angle"] == True).all(), key

    assert np.allclose(df["weight_decay"], wd), key
    assert np.allclose(df["dropout"], drop), key
    assert np.allclose(df["label_smoothing"], 0.0), key

    assert np.isfinite(
        df[["train_loss", "val_loss"]].to_numpy()
    ).all(), key

    cross_all_results[key] = df

    config_info.append({
        "Experiment": key,
        "Augmentation": aug,
        "Scheduler": sched,
        "WD": wd,
        "Dropout": drop,
    })

cross_summary = make_summary(
    cross_all_results
).merge(
    pd.DataFrame(config_info),
    on="Experiment",
    validate="one_to_one",
)

cross_summary = cross_summary[
    [
        "Experiment",
        "Augmentation",
        "Scheduler",
        "WD",
        "Dropout",
        "Mean Train Loss",
        "Std Train Loss",
        "Mean CV Loss",
        "Std CV Loss",
    ]
]

print("=== Cross Experiment Summary ===")
display(cross_summary.round(6))

cross_summary.to_csv(
    WORK_DIR / "cross_stage_experiment_summary.csv",
    index=False,
)

=== Cross Experiment Summary ===


,Experiment,Augmentation,Scheduler,WD,Dropout,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss
0,R_ON_BASE,rotation,True,0.0000,0.0,0.183117,0.069864,0.202408,0.053424
1,R_ON_WD,rotation,True,0.0001,0.0,0.204919,0.045626,0.206801,0.036108
2,R_OFF_BASE,rotation,False,0.0000,0.0,0.182455,0.037834,0.212584,0.043475
3,R_OFF_WD,rotation,False,0.0001,0.0,0.200961,0.017730,0.212759,0.044189
4,R_OFF_DROP,rotation,False,0.0000,0.4,0.226349,0.096712,0.213607,0.038488
5,C_ON_BASE,crop_light,True,0.0000,0.0,0.213186,0.031764,0.213942,0.042104
6,C_OFF_BASE,crop_light,False,0.0000,0.0,0.203321,0.034368,0.215460,0.023692
7,R_ON_DROP,rotation,True,0.0000,0.4,0.174263,0.060680,0.216965,0.045074
8,C_ON_WD,crop_light,True,0.0001,0.0,0.171216,0.050484,0.222092,0.035822
9,C_ON_DROP,crop_light,True,0.0000,0.4,0.183375,0.026464,0.223532,0.036357


In [54]:
# Five cross-stage interaction analyses

def loss_change(changed_key, reference_key):

    changed = (
        cross_all_results[changed_key]
        .set_index("fold")["val_loss"]
        .sort_index()
    )

    reference = (
        cross_all_results[reference_key]
        .set_index("fold")["val_loss"]
        .sort_index()
    )

    assert changed.index.equals(reference.index)

    return changed - reference


# Interaction design:
# (Name, Condition A, Effect A pair, Condition B, Effect B pair)

interaction_designs = [
    (
        "Augmentation x WD",
        "Rotation",
        ("R_ON_WD", "R_ON_BASE"),
        "Crop Light",
        ("C_ON_WD", "C_ON_BASE"),
    ),
    (
        "Augmentation x Dropout",
        "Rotation",
        ("R_ON_DROP", "R_ON_BASE"),
        "Crop Light",
        ("C_ON_DROP", "C_ON_BASE"),
    ),
    (
        "Augmentation x Scheduler",
        "Rotation",
        ("R_OFF_BASE", "R_ON_BASE"),
        "Crop Light",
        ("C_OFF_BASE", "C_ON_BASE"),
    ),
    (
        "Scheduler x WD",
        "Scheduler On",
        ("R_ON_WD", "R_ON_BASE"),
        "Scheduler Off",
        ("R_OFF_WD", "R_OFF_BASE"),
    ),
    (
        "Scheduler x Dropout",
        "Scheduler On",
        ("R_ON_DROP", "R_ON_BASE"),
        "Scheduler Off",
        ("R_OFF_DROP", "R_OFF_BASE"),
    ),
]

interaction_summary_rows = []
interaction_fold_rows = []

for name, label_a, pair_a, label_b, pair_b in interaction_designs:

    effect_a = loss_change(*pair_a)
    effect_b = loss_change(*pair_b)

    interaction = effect_b - effect_a

    interaction_summary_rows.append({
        "Interaction": name,
        "Condition A": label_a,
        "Effect A": effect_a.mean(),
        "Condition B": label_b,
        "Effect B": effect_b.mean(),
        "Interaction Estimate": interaction.mean(),
        "Fold Std": interaction.std(ddof=1),
        "Sign Reversal": bool(
            effect_a.mean() * effect_b.mean() < 0
        ),
    })

    for fold in interaction.index:

        interaction_fold_rows.append({
            "Interaction": name,
            "Fold": int(fold),
            "Effect A": effect_a.loc[fold],
            "Effect B": effect_b.loc[fold],
            "Interaction Estimate": interaction.loc[fold],
        })


interaction_summary = pd.DataFrame(
    interaction_summary_rows
)

interaction_fold_details = pd.DataFrame(
    interaction_fold_rows
)

print("=== Five Interaction Estimates ===")
display(interaction_summary.round(6))

print("\n=== Per-Fold Interaction Estimates ===")
display(interaction_fold_details.round(6))


interaction_summary.to_csv(
    WORK_DIR / "cross_stage_interaction_summary.csv",
    index=False,
)

interaction_fold_details.to_csv(
    WORK_DIR / "cross_stage_interaction_per_fold.csv",
    index=False,
)

print("All summaries saved to:", WORK_DIR)

=== Five Interaction Estimates ===


,Interaction,Condition A,Effect A,Condition B,Effect B,Interaction Estimate,Fold Std,Sign Reversal
0,Augmentation x WD,Rotation,0.004392,Crop Light,0.008150,0.003758,0.015773,False
1,Augmentation x Dropout,Rotation,0.014557,Crop Light,0.009590,-0.004967,0.031747,False
2,Augmentation x Scheduler,Rotation,0.010176,Crop Light,0.001519,-0.008657,0.028210,False
3,Scheduler x WD,Scheduler On,0.004392,Scheduler Off,0.000176,-0.004217,0.022071,False
4,Scheduler x Dropout,Scheduler On,0.014557,Scheduler Off,0.001023,-0.013534,0.033177,False



=== Per-Fold Interaction Estimates ===


,Interaction,Fold,Effect A,Effect B,Interaction Estimate
0,Augmentation x WD,0,0.018577,0.023292,0.004715
1,Augmentation x WD,1,0.018308,-0.002228,-0.020536
2,Augmentation x WD,2,-0.030711,-0.007543,0.023168
3,Augmentation x WD,3,0.006764,0.009353,0.002589
4,Augmentation x WD,4,0.009023,0.017877,0.008854
5,Augmentation x Dropout,0,0.020160,0.024549,0.004389
6,Augmentation x Dropout,1,0.044002,-0.004530,-0.048532
7,Augmentation x Dropout,2,0.001794,-0.007012,-0.008806
8,Augmentation x Dropout,3,0.020704,0.009192,-0.011512
9,Augmentation x Dropout,4,-0.013875,0.025750,0.039625


All summaries saved to: /kaggle/working/iceberg-data


In [55]:
!pip install -q smac==2.4.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [56]:

# Stage 6 / Cell A: search space and objective
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
from ConfigSpace import ConfigurationSpace, Categorical, Float, EqualsCondition
from smac import Scenario, HyperparameterOptimizationFacade as HPOFacade

# Must have executed the original model, dataloaders and CSV restore cell.
assert NUM_FOLDS == 5 and SEED == 42
assert USE_PRETRAINED is True

SMAC_SAVE_DIR = WORK_DIR / "stage6_smac"
SMAC_SAVE_DIR.mkdir(parents=True, exist_ok=True)

cs = ConfigurationSpace(seed=SEED)
aug = Categorical("augmentation", ["rotation", "crop_light", "baseline"], default="rotation")
optimizer_hp = Categorical("optimizer", ["adam", "adamw"], default="adam")
scheduler_hp = Categorical("scheduler", ["on", "off"], default="on")
lr_hp = Float("lr", (1e-4, 3e-3), log=True, default=1e-3)
wd_mode_hp = Categorical("wd_mode", ["off", "on"], default="off")
wd_value_hp = Float("wd_value", (1e-6, 1e-3), log=True, default=1e-4)
dropout_hp = Float("dropout", (0.0, 0.5), default=0.0)
smoothing_hp = Float("smoothing", (0.0, 0.15), default=0.0)
cs.add([aug, optimizer_hp, scheduler_hp, lr_hp,
        wd_mode_hp, wd_value_hp, dropout_hp, smoothing_hp])
cs.add(EqualsCondition(wd_value_hp, wd_mode_hp, "on"))

# The default configuration exactly reproduces R_ON_BASE.
print("Default SMAC config:", dict(cs.get_default_configuration()))

# Change in Cell B before searching: "smoke" for 3 trials, "full" for 30.
RUN_MODE = "smoke"


def config_to_train_args(config):
    x = dict(config)
    wd = float(x["wd_value"]) if x["wd_mode"] == "on" else 0.0
    return dict(
        augmentation=str(x["augmentation"]),
        optimizer_name=str(x["optimizer"]),
        use_scheduler=(x["scheduler"] == "on"),
        initial_lr=float(x["lr"]),
        weight_decay=wd,
        dropout_p=float(x["dropout"]),
        label_smoothing=float(x["smoothing"]),
    )


def smac_objective(config, seed=0):
    settings = config_to_train_args(config)
    # Hash the real training settings, avoiding filename collisions.
    serialized = json.dumps(settings, sort_keys=True, separators=(",", ":"))
    exp_id = hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:16]
    exp_name = f"stage6_smac_{exp_id}"

    results = run_5fold_experiment(
        **settings,
        model_name="efficientnet", use_angle=True, use_early_stopping=True,
        batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
        min_lr=MIN_LR, patience=PATIENCE,
        experiment_name=exp_name,
        save_models=False,
    )
    assert set(results["fold"].astype(int)) == set(range(NUM_FOLDS))
    fold_losses = results.sort_values("fold")["val_loss"].astype(float).tolist()
    score = float(np.mean(fold_losses))
    if not np.isfinite(score):
        raise ValueError(f"Non-finite CV loss for {exp_name}")

    record = {
        "experiment": exp_name, "mean_cv_loss": score,
        "std_cv_loss": float(np.std(fold_losses, ddof=1)),
        **settings,
        **{f"fold_{i}_loss": v for i, v in enumerate(fold_losses)},
    }
    log_path = SMAC_SAVE_DIR / f"{RUN_MODE}_trials.csv"
    old = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
    updated = pd.concat([old, pd.DataFrame([record])], ignore_index=True)
    updated = updated.drop_duplicates(subset=["experiment"], keep="last")
    temporary = log_path.with_suffix(".tmp")
    updated.to_csv(temporary, index=False)
    temporary.replace(log_path)
    print(f"[SMAC] {exp_name}: CV Loss = {score:.6f}", flush=True)
    return score


Default SMAC config: {'augmentation': 'rotation', 'dropout': 0.0, 'lr': 0.001, 'optimizer': 'adam', 'scheduler': 'on', 'smoothing': 0.0, 'wd_mode': 'off'}


In [57]:

# ============================================================
# Stage 6 / Cell B — SMAC Fresh Retry
# ============================================================

import json
import numpy as np
import pandas as pd

# First run 3 trials. Change to "full" for 30 trials.
RUN_MODE = "full"

# Allow NEW SMAC configurations to be trained.
RESTORE_ONLY = False

N_TRIALS = 3 if RUN_MODE == "smoke" else 30

# IMPORTANT:
# Use a NEW run name to avoid the previous failed history.
# Do NOT delete old results.
RUN_NAME = f"iceberg_stage6_{RUN_MODE}_retry1"

print("=== SMAC Configuration ===")
print("Mode:", RUN_MODE)
print("Number of trials:", N_TRIALS)
print("RESTORE_ONLY:", RESTORE_ONLY)
print("Run name:", RUN_NAME)

# ------------------------------------------------------------
# 1. Create SMAC scenario
# ------------------------------------------------------------

scenario = Scenario(
    configspace=cs,
    n_trials=N_TRIALS,
    seed=SEED,
    deterministic=True,
    use_default_config=True,
    n_workers=1,
    name=RUN_NAME,
    output_directory=SMAC_SAVE_DIR / "smac_state",
)

initial_design = HPOFacade.get_initial_design(
    scenario,
    n_configs=2 if RUN_MODE == "smoke" else 6,
    max_ratio=1.0,
)

intensifier = HPOFacade.get_intensifier(
    scenario,
    max_config_calls=1,
)

smac = HPOFacade(
    scenario,
    smac_objective,
    initial_design=initial_design,
    intensifier=intensifier,
    overwrite=False,
)

print("\nStarting SMAC optimization...")

# ------------------------------------------------------------
# 2. Run optimization
# ------------------------------------------------------------

best_config = smac.optimize()

history = smac.runhistory

print("\nFinished trials:", history.finished)

# ------------------------------------------------------------
# 3. Extract successful trial results
# ------------------------------------------------------------

records = []

for config in history.get_configs():

    score = float(history.get_cost(config))

    # Ignore failed configurations.
    if not np.isfinite(score):
        continue

    settings = config_to_train_args(config)

    records.append({
        "mean_cv_loss": score,
        **settings,
    })

if not records:
    print("\nTrial statuses:")
    for key, value in history.items():
        print(
            "Status:", value.status,
            "Cost:", value.cost,
            "Error:", (value.additional_info or {}).get("error")
        )

    raise RuntimeError(
        "SMAC has no successful trials. "
        "Inspect the errors printed above."
    )

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

trial_summary = pd.DataFrame(records)

trial_summary = trial_summary.sort_values(
    "mean_cv_loss"
).reset_index(drop=True)

print("\n=== SMAC Trial Results ===")
display(trial_summary.round(6))

best_args = config_to_train_args(best_config)
best_score = float(history.get_cost(best_config))

print("\n=== Best Parameters ===")
print(json.dumps(
    best_args,
    indent=2,
    ensure_ascii=False
))

print(f"\nBest SMAC CV Loss: {best_score:.6f}")
print("Previous R_ON_BASE CV Loss: 0.202408")

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

summary_path = (
    SMAC_SAVE_DIR /
    f"{RUN_NAME}_summary.csv"
)

best_path = (
    SMAC_SAVE_DIR /
    f"{RUN_NAME}_best.json"
)

trial_summary.to_csv(
    summary_path,
    index=False,
)

with best_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "best_parameters": best_args,
            "best_mean_cv_loss": best_score,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("\nSaved summary:", summary_path)
print("Saved best parameters:", best_path)


=== SMAC Configuration ===
Mode: full
Number of trials: 30
RESTORE_ONLY: False
Run name: iceberg_stage6_full_retry1
[INFO][abstract_initial_design.py:143] Using 6 initial design configurations and 1 additional configurations.
[INFO][smbo.py:523] Continuing from previous run.

Starting SMAC optimization...
[INFO][smbo.py:280] Optimization process was already finished. Returning incumbent...

Finished trials: 30

=== SMAC Trial Results ===


,mean_cv_loss,augmentation,optimizer_name,use_scheduler,initial_lr,weight_decay,dropout_p,label_smoothing
0,0.187174,rotation,adam,True,0.001354,0.000000,0.093327,0.003391
1,0.193193,rotation,adamw,True,0.001633,0.000371,0.413626,0.003228
2,0.197196,rotation,adam,True,0.001850,0.000000,0.033041,0.002881
3,0.202408,rotation,adam,True,0.001000,0.000000,0.000000,0.000000
4,0.205384,rotation,adamw,False,0.002884,0.000163,0.486658,0.003208
5,0.205940,rotation,adam,False,0.001431,0.000000,0.041732,0.001150
6,0.217274,crop_light,adam,False,0.001922,0.000168,0.309848,0.009651
7,0.217310,rotation,adam,False,0.001772,0.000405,0.065830,0.003150
8,0.221598,rotation,adam,True,0.000247,0.000000,0.130559,0.002744
9,0.221749,crop_light,adam,False,0.000988,0.000000,0.180319,0.040778



=== Best Parameters ===
{
  "augmentation": "rotation",
  "optimizer_name": "adam",
  "use_scheduler": true,
  "initial_lr": 0.0013541313614,
  "weight_decay": 0.0,
  "dropout_p": 0.0933272827915,
  "label_smoothing": 0.0033914502723
}

Best SMAC CV Loss: 0.187174
Previous R_ON_BASE CV Loss: 0.202408

Saved summary: /kaggle/working/iceberg-data/stage6_smac/iceberg_stage6_full_retry1_summary.csv
Saved best parameters: /kaggle/working/iceberg-data/stage6_smac/iceberg_stage6_full_retry1_best.json


In [58]:

# SMAC error diagnosis — NO TRAINING
from collections import Counter

print("=== 1. Training permission ===")
print("RESTORE_ONLY =", globals().get("RESTORE_ONLY", "NOT DEFINED"))

print("\n=== 2. SMAC trial history ===")

if "smac" not in globals():
    print("SMAC object is missing.")
else:
    history = smac.runhistory
    trials = list(history.items())

    print("Total recorded trials:", len(trials))
    print("Finished trials:", history.finished)

    statuses = Counter(
        str(value.status) for _, value in trials
    )
    print("Trial statuses:", dict(statuses))

    print("\n=== 3. Actual error messages ===")

    if not trials:
        print("RunHistory is empty. No trials were recorded.")
    else:
        for i, (key, value) in enumerate(trials[-5:], 1):
            print(f"\n--- Trial {i} ---")
            print("Status:", value.status)
            print("Cost:", value.cost)

            info = value.additional_info or {}
            print("Error:", info.get("error", "None"))

            tb = info.get("traceback", "")
            if tb:
                print("Traceback (last part):")
                print(tb[-2000:])


=== 1. Training permission ===
RESTORE_ONLY = False

=== 2. SMAC trial history ===
Total recorded trials: 30
Finished trials: 30
Trial statuses: {'1': 30}

=== 3. Actual error messages ===

--- Trial 1 ---
Status: 1
Cost: 0.20538367212219075
Error: None

--- Trial 2 ---
Status: 1
Cost: 0.23666861363428401
Error: None

--- Trial 3 ---
Status: 1
Cost: 0.24972061014690689
Error: None

--- Trial 4 ---
Status: 1
Cost: 0.25133829856506107
Error: None

--- Trial 5 ---
Status: 1
Cost: 0.22573292332464928
Error: None


# Version 2

In [64]:

# Version 2 - SMAC Optimized Final Model

FINAL_CONFIG = {
    "augmentation": "rotation",
    "model_name": "efficientnet",
    "optimizer_name": "adam",
    "initial_lr": 0.0013541313614,
    "weight_decay": 0.0,
    "dropout_p": 0.0933272827915,
    "label_smoothing": 0.0033914502723,
    "use_angle": True,
    "use_scheduler": True,
    "use_early_stopping": True,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "min_lr": MIN_LR,
    "patience": PATIENCE,
}

SMAC_BEST_CV_LOSS = 0.187174

# Each ablation changes only specified parameters.
def run_v2_ablation(name, **changes):
    config = FINAL_CONFIG.copy()

    unknown = set(changes) - set(config)
    if unknown:
        raise ValueError(f"Unknown parameters: {unknown}")

    config.update(changes)

    return run_5fold_experiment(
        experiment_name=f"v2_smac_{name}",
        save_models=False,
        **config,
    )


# Full model using the SMAC best configuration.
v2_full_results = run_v2_ablation("full_final")

full_cv_loss = mean_cv_loss(v2_full_results)

print("\n=== Version 2 Full Final Model ===")
print(f"Version 2 CV Loss: {full_cv_loss:.6f}")
print(f"SMAC Best CV Loss: {SMAC_BEST_CV_LOSS:.6f}")
print(
    f"Difference: "
    f"{full_cv_loss - SMAC_BEST_CV_LOSS:+.6f}"
)



=== v2_smac_full_final | efficientnet | rotation | adam | WD=0.0 | Dropout=0.0933272827915 | Smooth=0.0033914502723 | Angle=True | Scheduler=True | Early=True ===

========== Fold 0 ==========
Epoch 01 | Train Loss: 0.4877 | Val Loss: 0.3483 | LR: 0.001354
Epoch 02 | Train Loss: 0.3246 | Val Loss: 0.2823 | LR: 0.001350
Epoch 03 | Train Loss: 0.2890 | Val Loss: 0.3092 | LR: 0.001339
Epoch 04 | Train Loss: 0.2818 | Val Loss: 0.3085 | LR: 0.001321
Epoch 05 | Train Loss: 0.2543 | Val Loss: 0.5257 | LR: 0.001296
Epoch 06 | Train Loss: 0.2751 | Val Loss: 0.2780 | LR: 0.001264
Epoch 07 | Train Loss: 0.2576 | Val Loss: 0.3007 | LR: 0.001226
Epoch 08 | Train Loss: 0.2914 | Val Loss: 0.2676 | LR: 0.001182
Epoch 09 | Train Loss: 0.2292 | Val Loss: 0.3660 | LR: 0.001132
Epoch 10 | Train Loss: 0.2313 | Val Loss: 0.2004 | LR: 0.001077
Epoch 11 | Train Loss: 0.2365 | Val Loss: 0.2256 | LR: 0.001018
Epoch 12 | Train Loss: 0.2088 | Val Loss: 0.2554 | LR: 0.000955
Epoch 13 | Train Loss: 0.1889 | Val Lo

In [65]:

# Version 2 - Feature and Regularization Ablations

# 1. Remove Rotation
v2_no_rotation_results = run_v2_ablation(
    "no_rotation",
    augmentation="baseline",
)

# 2. Remove Dropout
v2_no_dropout_resuflts = run_v2_ablation(
    "no_dropout",
    dropout_p=0.0,
)

# 3. Remove Label Smoothing
v2_no_smoothing_results = run_v2_ablation(
    "no_label_smoothing",
    label_smoothing=0.0,
)

# 4. Remove Incidence Angle
v2_no_angle_results = run_v2_ablation(
    "no_angle",
    use_angle=False,
)

print("\nFeature and regularization ablations completed.")



=== v2_smac_no_rotation | efficientnet | baseline | adam | WD=0.0 | Dropout=0.0933272827915 | Smooth=0.0033914502723 | Angle=True | Scheduler=True | Early=True ===

========== Fold 0 ==========
Epoch 01 | Train Loss: 0.4591 | Val Loss: 1.1364 | LR: 0.001354
Epoch 02 | Train Loss: 0.3241 | Val Loss: 0.2841 | LR: 0.001350
Epoch 03 | Train Loss: 0.2675 | Val Loss: 0.2962 | LR: 0.001339
Epoch 04 | Train Loss: 0.2060 | Val Loss: 0.2870 | LR: 0.001321
Epoch 05 | Train Loss: 0.1477 | Val Loss: 0.6062 | LR: 0.001296
Epoch 06 | Train Loss: 0.2053 | Val Loss: 0.3616 | LR: 0.001264
Epoch 07 | Train Loss: 0.1810 | Val Loss: 0.3218 | LR: 0.001226
Epoch 08 | Train Loss: 0.1250 | Val Loss: 0.3932 | LR: 0.001182
Epoch 09 | Train Loss: 0.0705 | Val Loss: 0.3278 | LR: 0.001132
Epoch 10 | Train Loss: 0.0836 | Val Loss: 0.3796 | LR: 0.001077
Early stopping triggered.
Fold 0: selected epoch=2, train=0.3241, val=0.2841

========== Fold 1 ==========
Epoch 01 | Train Loss: 0.4859 | Val Loss: 0.5056 | LR: 0.0

In [66]:

# Version 2 - Training Strategy Ablations

# 5. Remove CosineAnnealingLR Scheduler
v2_no_scheduler_results = run_v2_ablation(
    "no_scheduler",
    use_scheduler=False,
)

# 6. Remove Early Stopping
v2_no_early_stopping_results = run_v2_ablation(
    "no_early_stopping",
    use_early_stopping=False,
)

# 7. Restore the original Learning Rate
v2_original_lr_results = run_v2_ablation(
    "original_lr",
    initial_lr=0.001,
)

print("\nTraining strategy ablations completed.")



=== v2_smac_no_scheduler | efficientnet | rotation | adam | WD=0.0 | Dropout=0.0933272827915 | Smooth=0.0033914502723 | Angle=True | Scheduler=False | Early=True ===

========== Fold 0 ==========
Epoch 01 | Train Loss: 0.4877 | Val Loss: 0.3483 | LR: 0.001354
Epoch 02 | Train Loss: 0.3244 | Val Loss: 0.2831 | LR: 0.001354
Epoch 03 | Train Loss: 0.2893 | Val Loss: 0.2893 | LR: 0.001354
Epoch 04 | Train Loss: 0.2839 | Val Loss: 0.2837 | LR: 0.001354
Epoch 05 | Train Loss: 0.2630 | Val Loss: 0.6720 | LR: 0.001354
Epoch 06 | Train Loss: 0.2566 | Val Loss: 0.2861 | LR: 0.001354
Epoch 07 | Train Loss: 0.2451 | Val Loss: 0.3431 | LR: 0.001354
Epoch 08 | Train Loss: 0.2833 | Val Loss: 0.2147 | LR: 0.001354
Epoch 09 | Train Loss: 0.2960 | Val Loss: 0.2501 | LR: 0.001354
Epoch 10 | Train Loss: 0.2722 | Val Loss: 0.2227 | LR: 0.001354
Epoch 11 | Train Loss: 0.2312 | Val Loss: 0.2130 | LR: 0.001354
Epoch 12 | Train Loss: 0.2229 | Val Loss: 0.2360 | LR: 0.001354
Epoch 13 | Train Loss: 0.1977 | Val

In [67]:

# Version 2 - Final Ablation Summary

import json

v2_results = {
    "Full Final Model": v2_full_results,
    "No Rotation": v2_no_rotation_results,
    "No Dropout": v2_no_dropout_results,
    "No Label Smoothing": v2_no_smoothing_results,
    "No Angle": v2_no_angle_results,
    "No Scheduler": v2_no_scheduler_results,
    "No Early Stopping": v2_no_early_stopping_results,
    "Original LR (0.001)": v2_original_lr_results,
}

# Validate the fold results.
for name, results in v2_results.items():
    assert len(results) == NUM_FOLDS, name
    assert set(results["fold"]) == set(range(NUM_FOLDS)), name

# Calculate five-fold summary statistics.
v2_summary = make_summary(v2_results)

full_cv_loss = mean_cv_loss(v2_full_results)

# Positive delta means worse performance.
v2_summary["Delta Mean CV Loss"] = (
    v2_summary["Mean CV Loss"] - full_cv_loss
)

v2_summary["Delta CV Loss (%)"] = (
    v2_summary["Delta Mean CV Loss"]
    / full_cv_loss * 100
)

print("=== Version 2 Final Ablation Summary ===")
display(v2_summary.round(6))

# Save summary.
WORK_DIR.mkdir(parents=True, exist_ok=True)

v2_summary.to_csv(
    WORK_DIR / "v2_final_ablation_summary.csv",
    index=False,
)

# Save per-fold results.
v2_fold_results = pd.concat(
    [
        results.assign(ablation=name)
        for name, results in v2_results.items()
    ],
    ignore_index=True,
)

v2_fold_results.to_csv(
    WORK_DIR / "v2_final_ablation_fold_results.csv",
    index=False,
)

# Save the final parameter configuration.
with open(
    WORK_DIR / "v2_smac_final_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(FINAL_CONFIG, f, indent=2)

print(f"\nFull Final Model CV Loss: {full_cv_loss:.6f}")
print(f"SMAC Search Best CV Loss: {SMAC_BEST_CV_LOSS:.6f}")
print(f"Results saved to: {WORK_DIR}")
print("Version 2 completed.")


=== Version 2 Final Ablation Summary ===


,Experiment,Mean Train Loss,Std Train Loss,Mean CV Loss,Std CV Loss,Delta Mean CV Loss,Delta CV Loss (%)
0,Full Final Model,0.185784,0.057092,0.187174,0.025398,0.000000,0.000000
1,No Scheduler,0.198230,0.056079,0.205306,0.021776,0.018132,9.687413
2,No Dropout,0.229285,0.090516,0.207358,0.023529,0.020185,10.783999
3,No Angle,0.164662,0.032881,0.211196,0.040309,0.024023,12.834372
4,No Label Smoothing,0.170724,0.053901,0.211286,0.031553,0.024113,12.882559
5,Original LR (0.001),0.243473,0.054772,0.213860,0.035932,0.026687,14.257708
6,No Rotation,0.191179,0.079817,0.237639,0.043898,0.050465,26.961871
7,No Early Stopping,0.041788,0.007126,0.270183,0.032489,0.083009,44.348858



Full Final Model CV Loss: 0.187174
SMAC Search Best CV Loss: 0.187174
Results saved to: /kaggle/working/iceberg-data
Version 2 completed.


In [68]:
from pathlib import Path
import gzip, pickle, os, json
import numpy as np
import pandas as pd
import torch

BACKUP_FILE = Path('/kaggle/working/iceberg-data/session_snapshot.pkl.gz')
BACKUP_FILE.parent.mkdir(parents=True, exist_ok=True)

required = ['train_df', 'train_ids', 'X_train_img', 'X_train_ang',
            'X_train_ang_missing', 'Y_train', 'fold_numbers']
missing = [x for x in required if x not in globals()]
assert not missing, f'Missing processed data: {missing}'
assert 'fold' in train_df.columns, 'train_df has no fold column'

def _snapshot_cpu(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu()
    if isinstance(x, dict):
        return {k: _snapshot_cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [_snapshot_cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple(_snapshot_cpu(v) for v in x)
    return x

saved, skipped = {}, {}
for name, value in list(globals().items()):
    if name.startswith('_') or name in {
        'In', 'Out', 'saved', 'skipped', 'BACKUP_FILE',
        'payload', 'manifest', 'snapshot', 'required',
        'missing', 'candidates', 'value', 'obj'
    }:
        continue

    if (
        isinstance(value, (
            pd.DataFrame, pd.Series, np.ndarray, torch.Tensor,
            dict, list, tuple
        ))
        or name in {'best_augmentation', 'EXPERIMENT_CACHE'}
    ):
        try:
            obj = _snapshot_cpu(value)
            if isinstance(obj, (dict, list, tuple)):
                pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
            saved[name] = obj
        except Exception as e:
            skipped[name] = str(e)

assert all(k in saved for k in required)

payload = {
    'variables': saved,
    'config': {
        k: globals().get(k)
        for k in [
            'SEED', 'BATCH_SIZE', 'NUM_FOLDS', 'MAX_EPOCHS',
            'INITIAL_LR', 'MIN_LR', 'PATIENCE', 'USE_PRETRAINED'
        ]
    },
}

temp = BACKUP_FILE.with_suffix('.tmp')
with gzip.open(temp, 'wb', compresslevel=1) as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

os.replace(temp, BACKUP_FILE)

manifest = {
    'saved_names': sorted(saved),
    'skipped': skipped,
    'size_mb': round(BACKUP_FILE.stat().st_size / 2**20, 2)
}
(BACKUP_FILE.parent / 'session_snapshot_manifest.json').write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print('Saved:', BACKUP_FILE)
print('Variables:', len(saved), 'Size:', manifest['size_mb'], 'MB')
print('Fold result variables:',
      sorted(n for n in saved if n.endswith('_results')))
print('Summary variables:',
      sorted(n for n in saved if n.endswith('_summary')))

if skipped:
    print('Skipped:', skipped)

Saved: /kaggle/working/iceberg-data/session_snapshot.pkl.gz
Variables: 111 Size: 153.18 MB
Fold result variables: ['augmentation_results', 'baseline_results', 'crop_light_results', 'crop_medium_results', 'cross_all_results', 'cross_crop_dropout_results', 'cross_crop_wd_results', 'cross_experiment_results', 'cross_new_results', 'cross_rotation_no_scheduler_results', 'cross_rotation_wd_no_scheduler_results', 'full_results', 'hflip_results', 'hvflip_results', 'no_early_stopping_results', 'no_scheduler_results', 'old_model_results', 'report_baseline_results', 'rotation_results', 'stage2_results', 'stage4_adam_dropout_results', 'stage4_adam_smoothing_dropout_results', 'stage4_adam_wd_dropout_results', 'stage4_adam_wd_results', 'stage4_adam_wd_smoothing_dropout_results', 'stage4_adam_wd_smoothing_results', 'stage4_adamw_dropout_results', 'stage4_adamw_no_wd_results', 'stage4_adamw_smoothing_dropout_results', 'stage4_adamw_smoothing_results', 'stage4_adamw_wd_dropout_results', 'stage4_adamw_w